# Лабораторная работа 4

Tensorflow 2.x

1) Подготовка данных

2) Использование Keras Model API

3) Использование Keras Sequential + Functional API

https://www.tensorflow.org/tutorials

Для выполнения лабораторной работы необходимо установить tensorflow версии 2.0 или выше .

Рекомендуется использовать возможности Colab'а по обучению моделей на GPU.



In [11]:
import os
import tensorflow as tf
import numpy as np
import math
import timeit
import matplotlib.pyplot as plt

%matplotlib inline

device = '/gpu:0' if tf.config.list_physical_devices('GPU') else '/cpu:0'
print(f"Using device: {device}")

Using device: /gpu:0


# Подготовка данных
Загрузите набор данных из предыдущей лабораторной работы.

In [12]:
def load_cifar10(num_training=49000, num_validation=1000, num_test=10000):
    """
    Fetch the CIFAR-10 dataset from the web and perform preprocessing to prepare
    it for the two-layer neural net classifier. These are the same steps as
    we used for the SVM, but condensed to a single function.
    """
    # Load the raw CIFAR-10 dataset and use appropriate data types and shapes
    cifar10 = tf.keras.datasets.cifar10.load_data()
    (X_train, y_train), (X_test, y_test) = cifar10
    X_train = np.asarray(X_train, dtype=np.float32)
    y_train = np.asarray(y_train, dtype=np.int32).flatten()
    X_test = np.asarray(X_test, dtype=np.float32)
    y_test = np.asarray(y_test, dtype=np.int32).flatten()

    # Subsample the data
    mask = range(num_training, num_training + num_validation)
    X_val = X_train[mask]
    y_val = y_train[mask]
    mask = range(num_training)
    X_train = X_train[mask]
    y_train = y_train[mask]
    mask = range(num_test)
    X_test = X_test[mask]
    y_test = y_test[mask]

    # Normalize the data: subtract the mean pixel and divide by std
    mean_pixel = X_train.mean(axis=(0, 1, 2), keepdims=True)
    std_pixel = X_train.std(axis=(0, 1, 2), keepdims=True)
    X_train = (X_train - mean_pixel) / std_pixel
    X_val = (X_val - mean_pixel) / std_pixel
    X_test = (X_test - mean_pixel) / std_pixel

    return X_train, y_train, X_val, y_val, X_test, y_test

# If there are errors with SSL downloading involving self-signed certificates,
# it may be that your Python version was recently installed on the current machine.
# See: https://github.com/tensorflow/tensorflow/issues/10779
# To fix, run the command: /Applications/Python\ 3.7/Install\ Certificates.command
#   ...replacing paths as necessary.

# Invoke the above function to get our data.
NHW = (0, 1, 2)
X_train, y_train, X_val, y_val, X_test, y_test = load_cifar10()
print('Train data shape: ', X_train.shape)
print('Train labels shape: ', y_train.shape, y_train.dtype)
print('Validation data shape: ', X_val.shape)
print('Validation labels shape: ', y_val.shape)
print('Test data shape: ', X_test.shape)
print('Test labels shape: ', y_test.shape)

Train data shape:  (49000, 32, 32, 3)
Train labels shape:  (49000,) int32
Validation data shape:  (1000, 32, 32, 3)
Validation labels shape:  (1000,)
Test data shape:  (10000, 32, 32, 3)
Test labels shape:  (10000,)


In [13]:
class Dataset(object):
    def __init__(self, X, y, batch_size, shuffle=False):
        """
        Construct a Dataset object to iterate over data X and labels y

        Inputs:
        - X: Numpy array of data, of any shape
        - y: Numpy array of labels, of any shape but with y.shape[0] == X.shape[0]
        - batch_size: Integer giving number of elements per minibatch
        - shuffle: (optional) Boolean, whether to shuffle the data on each epoch
        """
        assert X.shape[0] == y.shape[0], 'Got different numbers of data and labels'
        self.X, self.y = X, y
        self.batch_size, self.shuffle = batch_size, shuffle

    def __iter__(self):
        N, B = self.X.shape[0], self.batch_size
        idxs = np.arange(N)
        if self.shuffle:
            np.random.shuffle(idxs)
        return iter((self.X[i:i+B], self.y[i:i+B]) for i in range(0, N, B))


train_dset = Dataset(X_train, y_train, batch_size=64, shuffle=True)
val_dset = Dataset(X_val, y_val, batch_size=64, shuffle=False)
test_dset = Dataset(X_test, y_test, batch_size=64)

In [14]:
# We can iterate through a dataset like this:
for t, (x, y) in enumerate(train_dset):
    print(t, x.shape, y.shape)
    if t > 5: break

0 (64, 32, 32, 3) (64,)
1 (64, 32, 32, 3) (64,)
2 (64, 32, 32, 3) (64,)
3 (64, 32, 32, 3) (64,)
4 (64, 32, 32, 3) (64,)
5 (64, 32, 32, 3) (64,)
6 (64, 32, 32, 3) (64,)


#  Keras Model Subclassing API


Для реализации собственной модели с помощью Keras Model Subclassing API необходимо выполнить следующие шаги:

1) Определить новый класс, который является наследником tf.keras.Model.

2) В методе __init__() определить все необходимые слои из модуля tf.keras.layer

3) Реализовать прямой проход в методе call() на основе слоев, объявленных в __init__()

Ниже приведен пример использования keras API для определения двухслойной полносвязной сети.

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras

In [15]:
class TwoLayerFC(tf.keras.Model):
    def __init__(self, hidden_size, num_classes):
        super(TwoLayerFC, self).__init__()
        initializer = tf.initializers.VarianceScaling(scale=2.0)
        self.fc1 = tf.keras.layers.Dense(hidden_size, activation='relu',
                                   kernel_initializer=initializer)
        self.fc2 = tf.keras.layers.Dense(num_classes, activation='softmax',
                                   kernel_initializer=initializer)
        self.flatten = tf.keras.layers.Flatten()

    def call(self, x, training=False):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.fc2(x)
        return x


def test_TwoLayerFC():
    """ A small unit test to exercise the TwoLayerFC model above. """
    input_size, hidden_size, num_classes = 50, 42, 10
    x = tf.zeros((64, input_size))
    model = TwoLayerFC(hidden_size, num_classes)
    with tf.device(device):
        scores = model(x)
        print(scores.shape)

test_TwoLayerFC()

(64, 10)


Реализуйте трехслойную CNN для вашей задачи классификации.

Архитектура сети:
    
1. Сверточный слой (5 x 5 kernels, zero-padding = 'same')
2. Функция активации ReLU
3. Сверточный слой (3 x 3 kernels, zero-padding = 'same')
4. Функция активации ReLU
5. Полносвязный слой
6. Функция активации Softmax

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Conv2D

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Dense

In [16]:
class ThreeLayerConvNet(tf.keras.Model):
    def __init__(self, channel_1, channel_2, num_classes):
        super(ThreeLayerConvNet, self).__init__()
        ########################################################################
        # TODO: Implement the __init__ method for a three-layer ConvNet. You   #
        # should instantiate layer objects to be used in the forward pass.     #
        ########################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        initializer = tf.initializers.VarianceScaling(scale=2.0)

        self.conv1 = tf.keras.layers.Conv2D(channel_1, (5, 5), padding='same',
                                            activation='relu',
                                            kernel_initializer=initializer)

        self.conv2 = tf.keras.layers.Conv2D(channel_2, (3, 3), padding='same',
                                            activation='relu',
                                            kernel_initializer=initializer)

        self.flatten = tf.keras.layers.Flatten()
        self.fc = tf.keras.layers.Dense(num_classes, activation='softmax',
                                        kernel_initializer=initializer)

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ########################################################################
        #                           END OF YOUR CODE                           #
        ########################################################################

    def call(self, x, training=False):
        scores = None
        ########################################################################
        # TODO: Implement the forward pass for a three-layer ConvNet. You      #
        # should use the layer objects defined in the __init__ method.         #
        ########################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        x = self.conv1(x)
        x = self.conv2(x)
        x = self.flatten(x)
        scores = self.fc(x)

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ########################################################################
        #                           END OF YOUR CODE                           #
        ########################################################################
        return scores

In [17]:
def test_ThreeLayerConvNet():
    channel_1, channel_2, num_classes = 12, 8, 10
    model = ThreeLayerConvNet(channel_1, channel_2, num_classes)
    with tf.device(device):
        x = tf.zeros((64, 3, 32, 32))
        scores = model(x)
        print(scores.shape)

test_ThreeLayerConvNet()

(64, 10)


Пример реализации процесса обучения:

In [18]:
def train_part34(model_init_fn, optimizer_init_fn, num_epochs=1, is_training=False):
    """
    Simple training loop for use with models defined using tf.keras. It trains
    a model for one epoch on the CIFAR-10 training set and periodically checks
    accuracy on the CIFAR-10 validation set.

    Inputs:
    - model_init_fn: A function that takes no parameters; when called it
      constructs the model we want to train: model = model_init_fn()
    - optimizer_init_fn: A function which takes no parameters; when called it
      constructs the Optimizer object we will use to optimize the model:
      optimizer = optimizer_init_fn()
    - num_epochs: The number of epochs to train for

    Returns: Nothing, but prints progress during trainingn
    """
    with tf.device(device):


        loss_fn = tf.keras.losses.SparseCategoricalCrossentropy()

        model = model_init_fn()
        optimizer = optimizer_init_fn()

        train_loss = tf.keras.metrics.Mean(name='train_loss')
        train_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name='train_accuracy')

        val_loss = tf.keras.metrics.Mean(name='val_loss')
        val_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name='val_accuracy')

        t = 0
        for epoch in range(num_epochs):

            # Reset the metrics - https://www.tensorflow.org/alpha/guide/migration_guide#new-style_metrics
            train_loss.reset_state()
            train_accuracy.reset_state()

            for x_np, y_np in train_dset:
                with tf.GradientTape() as tape:

                    # Use the model function to build the forward pass.
                    scores = model(x_np, training=is_training)
                    loss = loss_fn(y_np, scores)

                    gradients = tape.gradient(loss, model.trainable_variables)
                    optimizer.apply_gradients(zip(gradients, model.trainable_variables))

                    # Update the metrics
                    train_loss.update_state(loss)
                    train_accuracy.update_state(y_np, scores)

                    if t % print_every == 0:
                        val_loss.reset_state()
                        val_accuracy.reset_state()
                        for test_x, test_y in val_dset:
                            # During validation at end of epoch, training set to False
                            prediction = model(test_x, training=False)
                            t_loss = loss_fn(test_y, prediction)

                            val_loss.update_state(t_loss)
                            val_accuracy.update_state(test_y, prediction)

                        template = 'Iteration {}, Epoch {}, Loss: {}, Accuracy: {}, Val Loss: {}, Val Accuracy: {}'
                        print (template.format(t, epoch+1,
                                             train_loss.result(),
                                             train_accuracy.result()*100,
                                             val_loss.result(),
                                             val_accuracy.result()*100))
                    t += 1

In [19]:
hidden_size, num_classes = 4000, 10
learning_rate = 1e-2

def model_init_fn():
    return TwoLayerFC(hidden_size, num_classes)

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate)

print_every = 100
train_part34(model_init_fn, optimizer_init_fn)

Iteration 0, Epoch 1, Loss: 2.5162858963012695, Accuracy: 17.1875, Val Loss: 2.7906546592712402, Val Accuracy: 13.300000190734863
Iteration 100, Epoch 1, Loss: 2.225883960723877, Accuracy: 29.42450714111328, Val Loss: 1.8640090227127075, Val Accuracy: 38.20000076293945
Iteration 200, Epoch 1, Loss: 2.0728347301483154, Accuracy: 32.633705139160156, Val Loss: 1.8651622533798218, Val Accuracy: 40.29999923706055
Iteration 300, Epoch 1, Loss: 2.0024118423461914, Accuracy: 34.35942840576172, Val Loss: 1.9401240348815918, Val Accuracy: 36.0
Iteration 400, Epoch 1, Loss: 1.937434196472168, Accuracy: 36.023223876953125, Val Loss: 1.7184727191925049, Val Accuracy: 42.20000076293945
Iteration 500, Epoch 1, Loss: 1.8914453983306885, Accuracy: 37.06337356567383, Val Loss: 1.6521251201629639, Val Accuracy: 43.20000076293945
Iteration 600, Epoch 1, Loss: 1.860121488571167, Accuracy: 38.07976150512695, Val Loss: 1.7057781219482422, Val Accuracy: 41.5
Iteration 700, Epoch 1, Loss: 1.8330553770065308, A

Обучите трехслойную CNN. В tf.keras.optimizers.SGD укажите Nesterov momentum = 0.9 .

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/optimizers/SGD

Значение accuracy на валидационной выборке после 1 эпохи обучения должно быть > 50% .

In [20]:
learning_rate = 3e-3
channel_1, channel_2, num_classes = 32, 16, 10

def model_init_fn():
    model = None
    ############################################################################
    # TODO: Complete the implementation of model_fn.                           #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    model = ThreeLayerConvNet(channel_1, channel_2, num_classes)

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                           END OF YOUR CODE                               #
    ############################################################################
    return model

def optimizer_init_fn():
    optimizer = None
    ############################################################################
    # TODO: Complete the implementation of model_fn.                           #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    optimizer = tf.keras.optimizers.SGD(learning_rate=learning_rate, momentum = 0.9, nesterov = True)

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                           END OF YOUR CODE                               #
    ############################################################################
    return optimizer

train_part34(model_init_fn, optimizer_init_fn)

Iteration 0, Epoch 1, Loss: 2.9287610054016113, Accuracy: 14.0625, Val Loss: 6.339081764221191, Val Accuracy: 8.399999618530273
Iteration 100, Epoch 1, Loss: 2.1740164756774902, Accuracy: 26.63985252380371, Val Loss: 1.83363938331604, Val Accuracy: 35.70000076293945
Iteration 200, Epoch 1, Loss: 1.9533681869506836, Accuracy: 32.33831024169922, Val Loss: 1.6275138854980469, Val Accuracy: 42.89999771118164
Iteration 300, Epoch 1, Loss: 1.8325316905975342, Accuracy: 36.08803939819336, Val Loss: 1.5169684886932373, Val Accuracy: 46.29999923706055
Iteration 400, Epoch 1, Loss: 1.7410047054290771, Accuracy: 38.76246643066406, Val Loss: 1.4591675996780396, Val Accuracy: 46.599998474121094
Iteration 500, Epoch 1, Loss: 1.6791260242462158, Accuracy: 40.75598907470703, Val Loss: 1.4250705242156982, Val Accuracy: 48.599998474121094
Iteration 600, Epoch 1, Loss: 1.6392186880111694, Accuracy: 42.12250518798828, Val Loss: 1.3870302438735962, Val Accuracy: 50.19999694824219
Iteration 700, Epoch 1, Lo

# Использование Keras Sequential API для реализации последовательных моделей.

Пример для полносвязной сети:

In [21]:
learning_rate = 1e-2

def model_init_fn():
    input_shape = (32, 32, 3)
    hidden_layer_size, num_classes = 4000, 10
    initializer = tf.initializers.VarianceScaling(scale=2.0)
    layers = [
        tf.keras.layers.Flatten(input_shape=input_shape),
        tf.keras.layers.Dense(hidden_layer_size, activation='relu',
                              kernel_initializer=initializer),
        tf.keras.layers.Dense(num_classes, activation='softmax',
                              kernel_initializer=initializer),
    ]
    model = tf.keras.Sequential(layers)
    return model

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate)

train_part34(model_init_fn, optimizer_init_fn)

Iteration 0, Epoch 1, Loss: 2.9596638679504395, Accuracy: 12.5, Val Loss: 2.9100818634033203, Val Accuracy: 14.30000114440918
Iteration 100, Epoch 1, Loss: 2.234400749206543, Accuracy: 28.04764747619629, Val Loss: 1.870964765548706, Val Accuracy: 37.900001525878906
Iteration 200, Epoch 1, Loss: 2.0756912231445312, Accuracy: 32.43159103393555, Val Loss: 1.8390135765075684, Val Accuracy: 39.599998474121094
Iteration 300, Epoch 1, Loss: 2.001351833343506, Accuracy: 34.35942840576172, Val Loss: 1.898433804512024, Val Accuracy: 37.29999923706055
Iteration 400, Epoch 1, Loss: 1.9338390827178955, Accuracy: 36.20246505737305, Val Loss: 1.7477432489395142, Val Accuracy: 42.89999771118164
Iteration 500, Epoch 1, Loss: 1.8892892599105835, Accuracy: 37.26921081542969, Val Loss: 1.6808394193649292, Val Accuracy: 42.29999923706055
Iteration 600, Epoch 1, Loss: 1.8590024709701538, Accuracy: 38.17595672607422, Val Loss: 1.6817407608032227, Val Accuracy: 42.5
Iteration 700, Epoch 1, Loss: 1.83312201499

Альтернативный менее гибкий способ обучения:

In [22]:
model = model_init_fn()
model.compile(optimizer=tf.keras.optimizers.SGD(learning_rate=learning_rate),
              loss='sparse_categorical_crossentropy',
              metrics=[tf.keras.metrics.sparse_categorical_accuracy])
model.fit(X_train, y_train, batch_size=64, epochs=1, validation_data=(X_val, y_val))
model.evaluate(X_test, y_test)

766/766 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - loss: 1.8292 - sparse_categorical_accuracy: 0.3879 - val_loss: 1.6200 - val_sparse_categorical_accuracy: 0.4330
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 1.6497 - sparse_categorical_accuracy: 0.4215


[1.6496587991714478, 0.42149999737739563]

Перепишите реализацию трехслойной CNN с помощью tf.keras.Sequential API . Обучите модель двумя способами.

In [23]:
def model_init_fn():
    model = None
    ############################################################################
    # TODO: Construct a three-layer ConvNet using tf.keras.Sequential.         #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(32, 32, 3)),
        tf.keras.layers.Conv2D(32, kernel_size=5, padding='same', activation='relu'),
        tf.keras.layers.Conv2D(16, kernel_size=3, padding='same', activation='relu'),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(10, activation='softmax'),
    ])

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                            END OF YOUR CODE                              #
    ############################################################################
    return model

learning_rate = 5e-4
def optimizer_init_fn():
    optimizer = None
    ############################################################################
    # TODO: Complete the implementation of model_fn.                           #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                           END OF YOUR CODE                               #
    ############################################################################
    return optimizer

train_part34(model_init_fn, optimizer_init_fn)

Iteration 0, Epoch 1, Loss: 2.2839815616607666, Accuracy: 20.3125, Val Loss: 2.512312412261963, Val Accuracy: 9.800000190734863
Iteration 100, Epoch 1, Loss: 1.840312123298645, Accuracy: 35.071163177490234, Val Loss: 1.5836421251296997, Val Accuracy: 45.5
Iteration 200, Epoch 1, Loss: 1.6982942819595337, Accuracy: 40.360694885253906, Val Loss: 1.456687092781067, Val Accuracy: 48.5
Iteration 300, Epoch 1, Loss: 1.6142821311950684, Accuracy: 43.09074020385742, Val Loss: 1.4035882949829102, Val Accuracy: 51.0
Iteration 400, Epoch 1, Loss: 1.5503324270248413, Accuracy: 45.44108581542969, Val Loss: 1.3498706817626953, Val Accuracy: 52.79999923706055
Iteration 500, Epoch 1, Loss: 1.5064667463302612, Accuracy: 46.91554260253906, Val Loss: 1.2950044870376587, Val Accuracy: 53.60000228881836
Iteration 600, Epoch 1, Loss: 1.4758174419403076, Accuracy: 48.044925689697266, Val Loss: 1.2614983320236206, Val Accuracy: 55.69999694824219
Iteration 700, Epoch 1, Loss: 1.4470778703689575, Accuracy: 49.2

In [24]:
model = model_init_fn()
model.compile(optimizer='sgd',
              loss='sparse_categorical_crossentropy',
              metrics=[tf.keras.metrics.sparse_categorical_accuracy])
model.fit(X_train, y_train, batch_size=64, epochs=1, validation_data=(X_val, y_val))
model.evaluate(X_test, y_test)

766/766 ━━━━━━━━━━━━━━━━━━━━ 7s 6ms/step - loss: 1.6454 - sparse_categorical_accuracy: 0.4175 - val_loss: 1.4218 - val_sparse_categorical_accuracy: 0.5010
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 1.4185 - sparse_categorical_accuracy: 0.4990


[1.4184730052947998, 0.49900001287460327]

# Использование Keras Functional API

Для реализации более сложных архитектур сети с несколькими входами/выходами, повторным использованием слоев, "остаточными" связями (residual connections) необходимо явно указать входные и выходные тензоры.

Ниже представлен пример для полносвязной сети.

In [25]:
def two_layer_fc_functional(input_shape, hidden_size, num_classes):
    initializer = tf.initializers.VarianceScaling(scale=2.0)
    inputs = tf.keras.Input(shape=input_shape)
    flattened_inputs = tf.keras.layers.Flatten()(inputs)
    fc1_output = tf.keras.layers.Dense(hidden_size, activation='relu',
                                 kernel_initializer=initializer)(flattened_inputs)
    scores = tf.keras.layers.Dense(num_classes, activation='softmax',
                             kernel_initializer=initializer)(fc1_output)

    # Instantiate the model given inputs and outputs.
    model = tf.keras.Model(inputs=inputs, outputs=scores)
    return model

def test_two_layer_fc_functional():
    """ A small unit test to exercise the TwoLayerFC model above. """
    input_size, hidden_size, num_classes = 50, 42, 10
    input_shape = (50,)

    x = tf.zeros((64, input_size))
    model = two_layer_fc_functional(input_shape, hidden_size, num_classes)

    with tf.device(device):
        scores = model(x)
        print(scores.shape)

test_two_layer_fc_functional()

(64, 10)


In [26]:
input_shape = (32, 32, 3)
hidden_size, num_classes = 4000, 10
learning_rate = 1e-2

def model_init_fn():
    return two_layer_fc_functional(input_shape, hidden_size, num_classes)

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate)

train_part34(model_init_fn, optimizer_init_fn)

Iteration 0, Epoch 1, Loss: 3.44985294342041, Accuracy: 4.6875, Val Loss: 2.96419358253479, Val Accuracy: 11.5
Iteration 100, Epoch 1, Loss: 2.2436106204986572, Accuracy: 28.001235961914062, Val Loss: 1.8976634740829468, Val Accuracy: 35.20000076293945
Iteration 200, Epoch 1, Loss: 2.078026533126831, Accuracy: 32.20615768432617, Val Loss: 1.8436576128005981, Val Accuracy: 39.89999771118164
Iteration 300, Epoch 1, Loss: 2.0002338886260986, Accuracy: 33.97010040283203, Val Loss: 1.8754024505615234, Val Accuracy: 36.89999771118164
Iteration 400, Epoch 1, Loss: 1.9288727045059204, Accuracy: 35.85956954956055, Val Loss: 1.7348490953445435, Val Accuracy: 41.29999923706055
Iteration 500, Epoch 1, Loss: 1.8843247890472412, Accuracy: 36.97292709350586, Val Loss: 1.6603261232376099, Val Accuracy: 42.5
Iteration 600, Epoch 1, Loss: 1.8529921770095825, Accuracy: 37.8743782043457, Val Loss: 1.670501708984375, Val Accuracy: 43.39999771118164
Iteration 700, Epoch 1, Loss: 1.8280644416809082, Accuracy

Поэкспериментируйте с архитектурой сверточной сети. Для вашего набора данных вам необходимо получить как минимум 70% accuracy на валидационной выборке за 10 эпох обучения. Опишите все эксперименты и сделайте выводы (без выполнения данного пункта работы приниматься не будут).

Эспериментируйте с архитектурой, гиперпараметрами, функцией потерь, регуляризацией, методом оптимизации.  

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/BatchNormalization#methods https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Dropout#methods

In [27]:
class CustomConvNet(tf.keras.Model):
    def __init__(self):
        super(CustomConvNet, self).__init__()
        ############################################################################
        # TODO: Construct a model that performs well on CIFAR-10                   #
        ############################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        initializer = tf.initializers.VarianceScaling(scale=2.0)

        # Блок 1
        self.conv1 = tf.keras.layers.Conv2D(64, 3, padding='same', kernel_initializer=initializer)
        self.bn1 = tf.keras.layers.BatchNormalization()
        self.dropout1 = tf.keras.layers.Dropout(0.25)

        # Блок 2
        self.conv2 = tf.keras.layers.Conv2D(64, 3, padding='same', kernel_initializer=initializer)
        self.bn2 = tf.keras.layers.BatchNormalization()
        self.pool1 = tf.keras.layers.MaxPooling2D(2)
        self.dropout2 = tf.keras.layers.Dropout(0.25)

        # Блок 3
        self.conv3 = tf.keras.layers.Conv2D(128, 3, padding='same', kernel_initializer=initializer)
        self.bn3 = tf.keras.layers.BatchNormalization()
        self.dropout3 = tf.keras.layers.Dropout(0.25)

        # Блок 4
        self.conv4 = tf.keras.layers.Conv2D(128, 3, padding='same', kernel_initializer=initializer)
        self.bn4 = tf.keras.layers.BatchNormalization()
        self.pool2 = tf.keras.layers.MaxPooling2D(2)
        self.dropout4 = tf.keras.layers.Dropout(0.25)

        # Классификатор
        self.flatten = tf.keras.layers.Flatten()
        self.fc1 = tf.keras.layers.Dense(512, activation='relu', kernel_initializer=initializer)
        self.dropout5 = tf.keras.layers.Dropout(0.5)
        self.fc2 = tf.keras.layers.Dense(10, activation='softmax', kernel_initializer=initializer)

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ############################################################################
        #                            END OF YOUR CODE                              #
        ############################################################################

    def call(self, input_tensor, training=False):
        ############################################################################
        # TODO: Construct a model that performs well on CIFAR-10                   #
        ############################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        x = self.conv1(input_tensor)
        x = self.bn1(x, training=training)
        x = tf.nn.relu(x)
        x = self.dropout1(x, training=training)

        x = self.conv2(x)
        x = self.bn2(x, training=training)
        x = tf.nn.relu(x)
        x = self.pool1(x)
        x = self.dropout2(x, training=training)

        x = self.conv3(x)
        x = self.bn3(x, training=training)
        x = tf.nn.relu(x)
        x = self.dropout3(x, training=training)

        x = self.conv4(x)
        x = self.bn4(x, training=training)
        x = tf.nn.relu(x)
        x = self.pool2(x)
        x = self.dropout4(x, training=training)

        x = self.flatten(x)
        x = self.fc1(x)
        x = self.dropout5(x, training=training)
        x = self.fc2(x)

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ############################################################################
        #                            END OF YOUR CODE                              #
        ############################################################################
        return x


print_every = 700
num_epochs = 10

model = CustomConvNet()

def model_init_fn():
    return CustomConvNet()

def optimizer_init_fn():
    learning_rate = 1e-3
    return tf.keras.optimizers.Adam(learning_rate)

train_part34(model_init_fn, optimizer_init_fn, num_epochs=num_epochs, is_training=True)

Iteration 0, Epoch 1, Loss: 5.449509620666504, Accuracy: 7.8125, Val Loss: 21.06505584716797, Val Accuracy: 11.5
Iteration 700, Epoch 1, Loss: 2.037353754043579, Accuracy: 29.47351837158203, Val Loss: 1.7260925769805908, Val Accuracy: 39.099998474121094
Iteration 1400, Epoch 2, Loss: 1.561333417892456, Accuracy: 41.23769760131836, Val Loss: 1.2855396270751953, Val Accuracy: 54.400001525878906
Iteration 2100, Epoch 3, Loss: 1.406153678894043, Accuracy: 47.218257904052734, Val Loss: 1.1679553985595703, Val Accuracy: 58.499996185302734
Iteration 2800, Epoch 4, Loss: 1.284972906112671, Accuracy: 52.44160461425781, Val Loss: 1.0996224880218506, Val Accuracy: 62.5
Iteration 3500, Epoch 5, Loss: 1.2143292427062988, Accuracy: 55.0700798034668, Val Loss: 0.999305009841919, Val Accuracy: 64.80000305175781
Iteration 4200, Epoch 6, Loss: 1.1638818979263306, Accuracy: 57.47978591918945, Val Loss: 0.8932771682739258, Val Accuracy: 69.5
Iteration 4900, Epoch 7, Loss: 1.103197693824768, Accuracy: 59.6

Опишите все эксперименты, результаты. Сделайте выводы.

Для достижения целевого показателя точности была спроектирована и реализована глубокая сверточная нейросеть CustomConvNet. В основу модели легла концепция последовательного извлечения признаков с постепенным увеличением количества фильтров и уменьшением пространственного разрешения данных.

Архитектура сети:

Conv2D(64, 3x3) -> BatchNorm -> ReLU -> Dropout(0.25)

Conv2D(64, 3x3) -> BatchNorm -> ReLU -> MaxPool(2x2) -> Dropout(0.25)

Conv2D(128, 3x3) -> BatchNorm -> ReLU -> Dropout(0.25)

Conv2D(128, 3x3) -> BatchNorm -> ReLU -> MaxPool(2x2) -> Dropout(0.25)

Flatten -> Dense(512) -> ReLU -> Dropout(0.5) -> Dense(10, softmax)

Высокая эффективность модели обеспечена комбинацией Batch Normalization, которая стабилизировала градиенты и ускорила сходимость, и слоев Dropout, предотвративших переобучение на малом наборе данных. Использование MaxPooling позволило сократить размерность и расширить поле зрения фильтров для анализа сложных паттернов. В сочетании с оптимизатором Adam (lr=1e-3) данная конфигурация позволила достичь >70% точности в рамках 10 эпох.